# EarningsLens — Stage 8a: Ollama Summarisation

For each company-quarter in the corpus, `qwen2.5:3b` generates a concise plain-English commitment summary grounded in the structured pipeline outputs — the extracted metric/value/timeframe slots, hedge scores, and cross-quarter statuses from Stages 4–6.

Summaries are generated from structured data, not raw transcript text, which prevents hallucination and keeps outputs factually anchored.

### Output

- `summaries.parquet` — one plain-English summary per company-quarter, consumed by Stage 10 dashboard

## 0. Dependencies

In [1]:
# %pip install -q ollama pandas tqdm

## 0.1 Imports & Logging

In [1]:
import os
import logging
from pathlib import Path
import psycopg2
from psycopg2.extras import execute_values
from dotenv import load_dotenv

import pandas as pd
from tqdm import tqdm
import ollama

In [2]:
logging.basicConfig(
    format="%(asctime)s [%(levelname)s] %(message)s",
    level=logging.INFO,
)
log = logging.getLogger("earningslens.stage8a")

print("\u2713 Imports OK")


✓ Imports OK


In [3]:
load_dotenv()
DATABASE_URL = os.getenv("DATABASE_URL")
if not DATABASE_URL:
    raise EnvironmentError("DATABASE_URL not found — check your .env file")

## 1. Configuration

In [4]:
# Paths
INPUT_PATH       = Path("fls_2023_2025.parquet")
CREDIBILITY_PATH = Path("credibility_summary.parquet")
SUMMARIES_PATH   = Path("summaries.parquet")

# Model
OLLAMA_MODEL     = "qwen2.5:3b"

# Limit per run — set to an integer (e.g. 500) to cap; None = all
# At ~3s/summary and ~4,700 company-quarters, full run is ~4 hrs
MAX_SUMMARIES    = None

print("\u2713 Configuration set")
print(f"  Ollama model  : {OLLAMA_MODEL}")
print(f"  Max summaries : {MAX_SUMMARIES or 'all'}")

✓ Configuration set
  Ollama model  : qwen2.5:3b
  Max summaries : all


## 2. Verify Ollama

In [5]:
try:
    test = ollama.chat(
        model    = OLLAMA_MODEL,
        messages = [{"role": "user", "content": "Reply with only: OK"}],
        options  = {"temperature": 0},
    )
    print(f"\u2713 Ollama responding: {test['message']['content'].strip()}")
except Exception as e:
    raise RuntimeError(
        f"Ollama not reachable: {e}\n"
        "Run 'ollama serve' in a terminal and ensure qwen2.5:3b is pulled."
    )

2026-05-07 04:01:32,036 [INFO] HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


✓ Ollama responding: OK


## 3. Load Data

In [6]:
fls         = pd.read_parquet(INPUT_PATH)
credibility = pd.read_parquet(CREDIBILITY_PATH)

In [7]:
log.info("Loaded %d FLS rows, %d credibility records", len(fls), len(credibility))

cred_lookup = credibility.set_index("ticker")["credibility"].to_dict()

print(f"FLS sentences        : {len(fls):,}")
print(f"Unique tickers       : {fls['ticker'].nunique():,}")
print(f"Credibility records  : {len(credibility):,}")
print(f"Unique company-qtrs  : {fls.groupby(['ticker','year','quarter']).ngroups:,}")

2026-05-07 04:01:32,299 [INFO] Loaded 92458 FLS rows, 500 credibility records


FLS sentences        : 92,458
Unique tickers       : 531
Credibility records  : 500
Unique company-qtrs  : 4,664


## 4. Summarisation Prompt & Functions

For each company-quarter the summariser receives a structured data block containing the top commitments (by FLS confidence) with their extracted slots, hedge score, and status. The LLM is instructed to produce a factual 3–5 sentence summary grounded entirely in the provided data. Temperature 0 ensures deterministic output.

In [8]:
SUMMARY_PROMPT = """You are a financial analyst assistant summarising management guidance from an earnings call.
Write a factual 3-5 sentence plain-English summary of the commitments below.
Use only the information provided. Do not speculate or add information not in the data.
Mention specific metrics, values, and timeframes where available.
Note the credibility score and whether commitments were delivered or missed in subsequent quarters.

Company: {company}
Quarter: Q{quarter} {year}
Credibility score: {credibility}
Mean hedge score: {hedge_score} (0=confident, 1=heavily hedged)
Overall tone: {tone}

Top commitments:
{commitments}

Summary:"""

In [9]:
def build_commitment_block(group: pd.DataFrame, max_rows: int = 8) -> str:
    """Format the top commitments for a company-quarter into a prompt block."""
    has_slots = group[group["slot_count"] > 0].copy()
    if has_slots.empty:
        has_slots = group.copy()

    top   = has_slots.nlargest(max_rows, "fls_confidence")
    lines = []
    for i, (_, row) in enumerate(top.iterrows(), 1):
        metric    = row.get("metric")    or "(not extracted)"
        value     = row.get("value")     or "(not extracted)"
        timeframe = row.get("timeframe") or "(not extracted)"
        hedge     = f"{row['hedge_score']:.2f}" if pd.notna(row.get("hedge_score")) else "N/A"
        status    = row.get("status")    or "Pending"
        sentence  = str(row["sentence"])[:150]
        lines.append(
            f"{i}. metric={metric} | value={value} | timeframe={timeframe} "
            f"| hedge={hedge} | status={status}\n"
            f"   '{sentence}'"
        )
    return "\n".join(lines)

In [10]:
def generate_summary(ticker: str, quarter: int, year: int, group: pd.DataFrame) -> str:
    """Generate a plain-English commitment summary for one company-quarter."""
    company_name = group["company_name"].iloc[0] if pd.notna(group["company_name"].iloc[0]) else ticker
    credibility  = cred_lookup.get(ticker)
    cred_str     = f"{credibility:.2f}" if credibility is not None else "N/A"
    mean_hedge   = group["hedge_score"].mean() if "hedge_score" in group.columns else None
    hedge_str    = f"{mean_hedge:.2f}" if mean_hedge is not None else "N/A"
    mean_finbert = group["finbert_score"].mean() if "finbert_score" in group.columns else 0
    tone_str     = "positive" if mean_finbert > 0.1 else ("negative" if mean_finbert < -0.1 else "neutral")

    prompt = SUMMARY_PROMPT.format(
        company     = company_name,
        quarter     = quarter,
        year        = year,
        credibility = cred_str,
        hedge_score = hedge_str,
        tone        = tone_str,
        commitments = build_commitment_block(group),
    )

    try:
        response = ollama.chat(
            model    = OLLAMA_MODEL,
            messages = [{"role": "user", "content": prompt}],
            options  = {"temperature": 0},
        )
        return response["message"]["content"].strip()
    except Exception as e:
        log.warning("Summary failed for %s Q%d %d: %s", ticker, quarter, year, e)
        return ""

## 5. Run Summarisation Loop

Summaries are generated for every company-quarter. The loop is checkpointed every 100 summaries — if interrupted, rerunning this cell picks up from where it left off.

In [11]:
# Load checkpoint if present
if SUMMARIES_PATH.exists():
    existing_summaries = pd.read_parquet(SUMMARIES_PATH)
    done_keys = set(
        existing_summaries["ticker"].astype(str) + "_" +
        existing_summaries["year"].astype(str) + "_Q" +
        existing_summaries["quarter"].astype(str)
    )
    log.info("Checkpoint: %d summaries already done", len(existing_summaries))
else:
    existing_summaries = pd.DataFrame()
    done_keys = set()

In [12]:
# Filter to remaining groups
groups = [
    g for g in fls.groupby(["ticker", "year", "quarter"])
    if f"{g[0][0]}_{g[0][1]}_Q{g[0][2]}" not in done_keys
]
if MAX_SUMMARIES:
    groups = groups[:MAX_SUMMARIES]

In [13]:
log.info(
    "Generating summaries for %d company-quarters — ~%.0f min at 3s/summary",
    len(groups), len(groups) * 3 / 60
)

summary_buffer = []

for (ticker, year, quarter), group in tqdm(groups, desc="Summarising"):
    summary_text = generate_summary(ticker, int(quarter), int(year), group)

    summary_buffer.append({
        "ticker"       : ticker,
        "company_name" : group["company_name"].iloc[0],
        "year"         : int(year),
        "quarter"      : int(quarter),
        "summary"      : summary_text,
        "n_commitments": len(group),
    })

    if len(summary_buffer) % 100 == 0:
        new_df   = pd.DataFrame(summary_buffer)
        combined = pd.concat([existing_summaries, new_df], ignore_index=True)
        combined.to_parquet(SUMMARIES_PATH, index=False)
        log.info("Checkpoint: %d summaries saved", len(combined))

2026-05-07 04:01:35,624 [INFO] Generating summaries for 4664 company-quarters — ~233 min at 3s/summary
Summarising:   2%|▏         | 99/4664 [03:13<2:14:12,  1.76s/it]2026-05-07 04:04:50,901 [INFO] HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-05-07 04:04:50,917 [INFO] Checkpoint: 100 summaries saved
Summarising:   4%|▍         | 199/4664 [06:46<2:56:21,  2.37s/it]2026-05-07 04:08:24,286 [INFO] HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-05-07 04:08:24,291 [INFO] Checkpoint: 200 summaries saved
Summarising:   6%|▋         | 299/4664 [10:03<2:21:38,  1.95s/it]2026-05-07 04:11:40,390 [INFO] HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-05-07 04:11:40,395 [INFO] Checkpoint: 300 summaries saved
Summarising:   9%|▊         | 399/4664 [13:16<2:11:35,  1.85s/it]2026-05-07 04:14:53,289 [INFO] HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-05-07 04:14:53,294 [INFO] Checkpoint: 400 su

In [18]:
summary_buffer

[{'ticker': 'A',
  'company_name': 'Agilent Technologies, Inc.',
  'year': 2023,
  'quarter': 1,
  'summary': 'In Q1 2023, Agilent Technologies, Inc. provided guidance on several key metrics. The company forecasted non-GAAP earnings per share to be between $1.24 and $1.27 for the second quarter, representing a growth of 10% to 12% compared to the previous year, which was delivered. Revenue expectations were set at $1.655 billion to $1.680 billion for Q2, with currency expected to be a headwind of 3.1 points in that quarter. The company also announced plans for a mid-calendar year manufacturing expansion and an update on capital spending, which was later withdrawn. Currency remained a significant factor, contributing as a 300 basis point headwind for the full year.',
  'n_commitments': 13},
 {'ticker': 'A',
  'company_name': 'Agilent Technologies, Inc.',
  'year': 2023,
  'quarter': 2,
  'summary': "Agilent Technologies' management guidance for the second quarter of 2023 includes commit

In [19]:
# Final save
if summary_buffer:
    new_df       = pd.DataFrame(summary_buffer)
    combined     = pd.concat([existing_summaries, new_df], ignore_index=True)
    combined.to_parquet(SUMMARIES_PATH, index=False)
    summaries_df = combined
else:
    summaries_df = existing_summaries

log.info("Summarisation complete: %d summaries", len(summaries_df))
print(f"\n\u2713 Summaries complete: {len(summaries_df):,}")

2026-05-07 07:08:07,244 [INFO] Summarisation complete: 4664 summaries



✓ Summaries complete: 4,664


## 6. Validate Summaries

In [20]:
print("\u2500\u2500 Sample generated summaries \u2500\u2500\n")
sample = summaries_df.dropna(subset=["summary"]).sample(min(4, len(summaries_df)), random_state=42)
for _, row in sample.iterrows():
    print(f"  {row['ticker']} \u2014 Q{row['quarter']} {row['year']} ({row['n_commitments']} commitments)")
    print(f"  {row['summary']}")
    print()

── Sample generated summaries ──

  MCO — Q1 2023 (15 commitments)
  In Moody's Corporation's Q1 2023 earnings call, management provided guidance on various financial metrics. For incentive compensation, they expect it to be between $340 million and $360 million for the full year 2023 with a credibility score of 0.65. The company aims for first half MIS revenue to decline in the mid- to high single-digit percent range and second half growth in the mid to high teens percent range, though this was rated as Pending. Moody's plans to return approximately $800 million of global free cash flow by year-end with a credibility score of 0.64. The company expects operating expenses to increase at the lower end of the mid-single-digit range for full-year 2023 (status withdrawn). For sales growth, they anticipate it to be in the high single digit this year (status withdrawn). Moody's revised its EPS guidance to between $8.45 and $8.95 for GAAP diluted EPS and $9.50 and $10 for adjusted diluted EPS 

In [21]:
summaries_df = pd.read_parquet("summaries.parquet")

In [22]:
conn   = psycopg2.connect(DATABASE_URL + "?sslmode=require")
cursor = conn.cursor()

summary_rows = [
    (row["ticker"], row["company_name"], row["quarter"], row["year"], row["summary"], row["n_commitments"])
    for _, row in summaries_df.iterrows()
]

execute_values(
    cursor,
    """
    INSERT INTO summaries (ticker, company_name, quarter, year, summary_text, n_commitments)
    VALUES %s
    ON CONFLICT DO NOTHING
    """,
    summary_rows,
    page_size=500
)
conn.commit()
conn.close()
print(f"✓ Stage 8a — {len(summary_rows):,} summaries written to Supabase")

✓ Stage 8a — 4,664 summaries written to Supabase


## Stage 8a Summary

In [ ]:
print("=" * 52)
print("  EarningsLens \u2014 Stage 8a Complete")
print("=" * 52)
print(f"\nSummaries generated  : {len(summaries_df):,}")
print(f"Saved to             : {SUMMARIES_PATH}")
empty = (summaries_df["summary"] == "").sum()
print(f"Failed (empty)       : {empty:,}")
print(f"\n\u2192 Ready for Stage 8b \u2014 RAG Pipeline")

  EarningsLens — Stage 8a Complete

Summaries generated  : 4,664
Saved to             : summaries.parquet
Failed (empty)       : 0

→ Ready for Stage 8b — RAG Pipeline


: 